## Agenda

1. Recall: DATA step processing and the PDV
2. `DROP`, `WHERE`, and `IF` in the PDV
3. `_ERROR_`
4. `RETAIN` with BY groups: `FIRST.` and `LAST.`
5. Merging tables

## Recall

| | |
|---|---|
| **DATA step processing** | 2 phases: compilation and execution |
| **PDV** | the program data vector, a temporary area of memory that SAS uses inside a DATA step to build one row at a time |

::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
In compilation, SAS checks the syntax, figures out the columns and their attributes, and builds the PDV. No data is read yet. In execution, SAS reads one row at a time, runs the statements, and writes the row out.

At the start of each new row, new columns are reset to missing, unless we `RETAIN` them.
:::


## A full PDV example

Let's walk through one DATA step from start to finish. Here is a small table with 5 rows.

In [ ]:
data scores;
    input Student $ Quiz1 Quiz2;
    datalines;
Vince 8 9
Abi 6 7
Jon 9 10
Maria 7 7
Kristian 10 .
;
run;

```{=html}
<table class="mini"><tr><th>Student</th><th>Quiz1</th><th>Quiz2</th></tr><tr><td>Vince</td><td>8</td><td>9</td></tr><tr><td>Abi</td><td>6</td><td>7</td></tr><tr><td>Jon</td><td>9</td><td>10</td></tr><tr><td>Maria</td><td>7</td><td>7</td></tr><tr><td>Kristian</td><td>10</td><td>.</td></tr></table>
```

::: {.content-visible unless-format="revealjs"}
`datalines` lets us type a small table right into our code. Kristian missed the second quiz, so his `Quiz2` is a period, which is how we write a missing number.
:::

## The code

We want each student's total, whether they passed (15 or more), and a running total of everyone's points.

In [ ]:
data quiz_totals;
    set scores;
    retain Running 0;
    Total = Quiz1 + Quiz2;
    if Total >= 15 then Pass = "Y";
    else Pass = "N";
    Running = sum(Running, Total);
    drop Quiz1 Quiz2;
run;

::: {.content-visible unless-format="revealjs"}
Before reading on, try to guess what the output table will look like. Which columns will it have? What will Kristian's row say?
:::

::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
This one step uses almost everything we've learned so far: `SET`, `RETAIN`, an assignment, `IF`/`ELSE`, `sum()`, and `DROP`.
:::


## Compilation

SAS reads the code top to bottom and builds the PDV. No data is read yet.

```{=html}
<div class="pdv-walk">
<div class="pdv-side"><pre class="pdv-code"><span class="pdv-line"><span class="kw">data</span> quiz_totals;<span class="fragment pdv-lit" data-fragment-index="1"></span></span><span class="pdv-line">    <span class="kw">set</span> scores;<span class="fragment pdv-lit" data-fragment-index="2"></span></span><span class="pdv-line">    <span class="kw">retain</span> Running 0;<span class="fragment pdv-lit" data-fragment-index="3"></span></span><span class="pdv-line">    Total = Quiz1 + Quiz2;<span class="fragment pdv-lit" data-fragment-index="4"></span></span><span class="pdv-line">    <span class="kw">if</span> Total &gt;= 15 <span class="kw">then</span> Pass = "Y";<span class="fragment pdv-lit" data-fragment-index="5"></span></span><span class="pdv-line">    <span class="kw">else</span> Pass = "N";</span><span class="pdv-line">    Running = <span class="kw">sum</span>(Running, Total);</span><span class="pdv-line">    <span class="kw">drop</span> Quiz1 Quiz2;<span class="fragment pdv-lit" data-fragment-index="6"></span></span><span class="pdv-line"><span class="kw">run</span>;</span></pre></div>
<div class="pdv-wrap"><table class="pdv"><thead><tr><th class="fragment keep" data-fragment-index="1">_N_<span class="pdv-tag fragment" data-fragment-index="6">D</span></th><th class="fragment keep" data-fragment-index="1">_ERROR_<span class="pdv-tag fragment" data-fragment-index="6">D</span></th><th class="fragment keep" data-fragment-index="2">Student<span class="pdv-tag fragment" data-fragment-index="6">K</span></th><th class="fragment keep" data-fragment-index="2">Quiz1<span class="pdv-tag fragment" data-fragment-index="6">D</span></th><th class="fragment keep" data-fragment-index="2">Quiz2<span class="pdv-tag fragment" data-fragment-index="6">D</span></th><th class="fragment keep" data-fragment-index="3">Running<span class="pdv-tag fragment" data-fragment-index="6">K</span></th><th class="fragment keep" data-fragment-index="4">Total<span class="pdv-tag fragment" data-fragment-index="6">K</span></th><th class="fragment keep" data-fragment-index="5">Pass<span class="pdv-tag fragment" data-fragment-index="6">K</span></th></tr></thead></table></div>
</div>
```

::: {.fragment}
`SET` adds the columns from `scores`. `RETAIN` adds `Running`, and sets it to 0. The next two lines add `Total` (numeric) and `Pass` (character, length 1 from `"Y"`). `DROP` marks `Quiz1` and `Quiz2` with a `D`.
:::


::: {.content-visible unless-format="revealjs"}
`RETAIN` and `DROP` do all their work here, during compilation. During execution, SAS skips right over them.

`Pass` gets a length of 1 because the first value SAS sees for it is `"Y"`, which is one character long.

::: {.callout-tip collapse="true" title="Check yourself: which columns end up in the output table?"}
`Student`, `Running`, `Total`, and `Pass`. Everything marked `K`, in the order they were added to the PDV.
:::
:::

## Execution: row 1

`_N_` is 1. `Running` starts at 0 because of `RETAIN`.

```{=html}
<div class="pdv-walk">
<div class="pdv-side"><pre class="pdv-code"><span class="pdv-line"><span class="kw">data</span> quiz_totals;<span class="fragment pdv-lit" data-fragment-index="1"></span></span><span class="pdv-line">    <span class="kw">set</span> scores;<span class="fragment pdv-lit" data-fragment-index="2"></span></span><span class="pdv-line">    <span class="kw">retain</span> Running 0;</span><span class="pdv-line">    Total = Quiz1 + Quiz2;<span class="fragment pdv-lit" data-fragment-index="3"></span></span><span class="pdv-line">    <span class="kw">if</span> Total &gt;= 15 <span class="kw">then</span> Pass = "Y";<span class="fragment pdv-lit" data-fragment-index="4"></span></span><span class="pdv-line">    <span class="kw">else</span> Pass = "N";</span><span class="pdv-line">    Running = <span class="kw">sum</span>(Running, Total);<span class="fragment pdv-lit" data-fragment-index="5"></span></span><span class="pdv-line">    <span class="kw">drop</span> Quiz1 Quiz2;</span><span class="pdv-line"><span class="kw">run</span>;<span class="fragment pdv-lit" data-fragment-index="6"></span></span></pre><div class="pdv-in"><div class="pdv-cap">Input: <code>work.scores</code></div><table class="mini pdv-intab"><tr><th>Student</th><th>Quiz1</th><th>Quiz2</th></tr><tr class="fragment pdv-inrow" data-fragment-index="2"><td>Vince</td><td>8</td><td>9</td></tr><tr><td>Abi</td><td>6</td><td>7</td></tr><tr><td>Jon</td><td>9</td><td>10</td></tr><tr><td>Maria</td><td>7</td><td>7</td></tr><tr><td>Kristian</td><td>10</td><td>.</td></tr></table></div><div class="out-table"><div class="pdv-cap">Output: <code>work.quiz_totals</code></div><table class="mini"><tr><th>Student</th><th>Running</th><th>Total</th><th>Pass</th></tr><tr class="fragment pdv-out-new" data-fragment-index="6"><td class="xfer">Vince</td><td class="xfer">17</td><td class="xfer">17</td><td class="xfer">Y</td></tr></table></div></div>
<div class="pdv-wrap"><table class="pdv"><tr><th class="drop">_N_<span class="pdv-tag">D</span></th><th class="drop">_ERROR_<span class="pdv-tag">D</span></th><th class="keep">Student<span class="pdv-tag">K</span></th><th class="drop">Quiz1<span class="pdv-tag">D</span></th><th class="drop">Quiz2<span class="pdv-tag">D</span></th><th class="keep">Running<span class="pdv-tag">K</span></th><th class="keep">Total<span class="pdv-tag">K</span></th><th class="keep">Pass<span class="pdv-tag">K</span></th></tr><tr class="fragment" data-fragment-index="1"><td class="hl">1</td><td>0</td><td></td><td>.</td><td>.</td><td>0</td><td>.</td><td></td></tr><tr class="fragment" data-fragment-index="2"><td>1</td><td>0</td><td class="hl">Vince</td><td class="hl">8</td><td class="hl">9</td><td>0</td><td>.</td><td></td></tr><tr class="fragment" data-fragment-index="3"><td>1</td><td>0</td><td>Vince</td><td>8</td><td>9</td><td>0</td><td class="hl">17</td><td></td></tr><tr class="fragment" data-fragment-index="4"><td>1</td><td>0</td><td>Vince</td><td>8</td><td>9</td><td>0</td><td>17</td><td class="hl">Y</td></tr><tr class="fragment pdv-final" data-fragment-index="5"><td>1</td><td>0</td><td class="kc">Vince</td><td>8</td><td>9</td><td class="hl kc">17</td><td class="kc">17</td><td class="kc">Y</td></tr></table></div>
</div>
```

::: {.fragment}
`Total` is 17, so the `IF` is true and SAS skips the `ELSE`. At `run;`, SAS writes the row to the output table, without `_N_`, `_ERROR_`, `Quiz1`, and `Quiz2`.
:::

::: {.content-visible unless-format="revealjs"}
Notice that `RETAIN` and `DROP` are never highlighted. They aren't executable, so SAS doesn't run them row by row.
:::

## Execution: row 2

Back to the top. `Total` and `Pass` reset to missing, but `Running` keeps its value.

```{=html}
<div class="pdv-walk">
<div class="pdv-side"><pre class="pdv-code"><span class="pdv-line"><span class="kw">data</span> quiz_totals;<span class="fragment pdv-lit" data-fragment-index="1"></span></span><span class="pdv-line">    <span class="kw">set</span> scores;<span class="fragment pdv-lit" data-fragment-index="2"></span></span><span class="pdv-line">    <span class="kw">retain</span> Running 0;</span><span class="pdv-line">    Total = Quiz1 + Quiz2;<span class="fragment pdv-lit" data-fragment-index="3"></span></span><span class="pdv-line">    <span class="kw">if</span> Total &gt;= 15 <span class="kw">then</span> Pass = "Y";<span class="fragment pdv-lit" data-fragment-index="4"></span></span><span class="pdv-line">    <span class="kw">else</span> Pass = "N";<span class="fragment pdv-lit" data-fragment-index="5"></span></span><span class="pdv-line">    Running = <span class="kw">sum</span>(Running, Total);<span class="fragment pdv-lit" data-fragment-index="6"></span></span><span class="pdv-line">    <span class="kw">drop</span> Quiz1 Quiz2;</span><span class="pdv-line"><span class="kw">run</span>;<span class="fragment pdv-lit" data-fragment-index="7"></span></span></pre><div class="pdv-in"><div class="pdv-cap">Input: <code>work.scores</code></div><table class="mini pdv-intab"><tr><th>Student</th><th>Quiz1</th><th>Quiz2</th></tr><tr><td>Vince</td><td>8</td><td>9</td></tr><tr class="fragment pdv-inrow" data-fragment-index="2"><td>Abi</td><td>6</td><td>7</td></tr><tr><td>Jon</td><td>9</td><td>10</td></tr><tr><td>Maria</td><td>7</td><td>7</td></tr><tr><td>Kristian</td><td>10</td><td>.</td></tr></table></div><div class="out-table"><div class="pdv-cap">Output: <code>work.quiz_totals</code></div><table class="mini"><tr><th>Student</th><th>Running</th><th>Total</th><th>Pass</th></tr><tr><td>Vince</td><td>17</td><td>17</td><td>Y</td></tr><tr class="fragment pdv-out-new" data-fragment-index="7"><td class="xfer">Abi</td><td class="xfer">30</td><td class="xfer">13</td><td class="xfer">N</td></tr></table></div></div>
<div class="pdv-wrap"><table class="pdv"><tr><th class="drop">_N_<span class="pdv-tag">D</span></th><th class="drop">_ERROR_<span class="pdv-tag">D</span></th><th class="keep">Student<span class="pdv-tag">K</span></th><th class="drop">Quiz1<span class="pdv-tag">D</span></th><th class="drop">Quiz2<span class="pdv-tag">D</span></th><th class="keep">Running<span class="pdv-tag">K</span></th><th class="keep">Total<span class="pdv-tag">K</span></th><th class="keep">Pass<span class="pdv-tag">K</span></th></tr><tr class="fragment" data-fragment-index="1"><td class="hl">2</td><td>0</td><td>Vince</td><td>8</td><td>9</td><td>17</td><td class="hl">.</td><td class="hl"></td></tr><tr class="fragment" data-fragment-index="2"><td>2</td><td>0</td><td class="hl">Abi</td><td class="hl">6</td><td class="hl">7</td><td>17</td><td>.</td><td></td></tr><tr class="fragment" data-fragment-index="3"><td>2</td><td>0</td><td>Abi</td><td>6</td><td>7</td><td>17</td><td class="hl">13</td><td></td></tr><tr class="fragment" data-fragment-index="4"><td>2</td><td>0</td><td>Abi</td><td>6</td><td>7</td><td>17</td><td class="hl">13</td><td></td></tr><tr class="fragment" data-fragment-index="5"><td>2</td><td>0</td><td>Abi</td><td>6</td><td>7</td><td>17</td><td>13</td><td class="hl">N</td></tr><tr class="fragment pdv-final" data-fragment-index="6"><td>2</td><td>0</td><td class="kc">Abi</td><td>6</td><td>7</td><td class="hl kc">30</td><td class="kc">13</td><td class="kc">N</td></tr></table></div>
</div>
```

::: {.fragment}
`Total` is 13, so the `IF` is false and the `ELSE` runs.
:::

::: {.content-visible unless-format="revealjs"}
`Student`, `Quiz1`, and `Quiz2` still hold Vince's values at the top of the step. That's fine, because `SET` replaces them right away. `Total` and `Pass` are new columns, so SAS sets them back to missing. `Running` is retained, so it keeps 17.
:::

## Execution: row 3

Same steps, next row.

```{=html}
<div class="pdv-walk">
<div class="pdv-side"><pre class="pdv-code"><span class="pdv-line"><span class="kw">data</span> quiz_totals;<span class="fragment pdv-lit" data-fragment-index="1"></span></span><span class="pdv-line">    <span class="kw">set</span> scores;<span class="fragment pdv-lit" data-fragment-index="2"></span></span><span class="pdv-line">    <span class="kw">retain</span> Running 0;</span><span class="pdv-line">    Total = Quiz1 + Quiz2;<span class="fragment pdv-lit" data-fragment-index="3"></span></span><span class="pdv-line">    <span class="kw">if</span> Total &gt;= 15 <span class="kw">then</span> Pass = "Y";<span class="fragment pdv-lit" data-fragment-index="4"></span></span><span class="pdv-line">    <span class="kw">else</span> Pass = "N";</span><span class="pdv-line">    Running = <span class="kw">sum</span>(Running, Total);<span class="fragment pdv-lit" data-fragment-index="5"></span></span><span class="pdv-line">    <span class="kw">drop</span> Quiz1 Quiz2;</span><span class="pdv-line"><span class="kw">run</span>;<span class="fragment pdv-lit" data-fragment-index="6"></span></span></pre><div class="pdv-in"><div class="pdv-cap">Input: <code>work.scores</code></div><table class="mini pdv-intab"><tr><th>Student</th><th>Quiz1</th><th>Quiz2</th></tr><tr><td>Vince</td><td>8</td><td>9</td></tr><tr><td>Abi</td><td>6</td><td>7</td></tr><tr class="fragment pdv-inrow" data-fragment-index="2"><td>Jon</td><td>9</td><td>10</td></tr><tr><td>Maria</td><td>7</td><td>7</td></tr><tr><td>Kristian</td><td>10</td><td>.</td></tr></table></div><div class="out-table"><div class="pdv-cap">Output: <code>work.quiz_totals</code></div><table class="mini"><tr><th>Student</th><th>Running</th><th>Total</th><th>Pass</th></tr><tr><td>Vince</td><td>17</td><td>17</td><td>Y</td></tr><tr><td>Abi</td><td>30</td><td>13</td><td>N</td></tr><tr class="fragment pdv-out-new" data-fragment-index="6"><td class="xfer">Jon</td><td class="xfer">49</td><td class="xfer">19</td><td class="xfer">Y</td></tr></table></div></div>
<div class="pdv-wrap"><table class="pdv"><tr><th class="drop">_N_<span class="pdv-tag">D</span></th><th class="drop">_ERROR_<span class="pdv-tag">D</span></th><th class="keep">Student<span class="pdv-tag">K</span></th><th class="drop">Quiz1<span class="pdv-tag">D</span></th><th class="drop">Quiz2<span class="pdv-tag">D</span></th><th class="keep">Running<span class="pdv-tag">K</span></th><th class="keep">Total<span class="pdv-tag">K</span></th><th class="keep">Pass<span class="pdv-tag">K</span></th></tr><tr class="fragment" data-fragment-index="1"><td class="hl">3</td><td>0</td><td>Abi</td><td>6</td><td>7</td><td>30</td><td class="hl">.</td><td class="hl"></td></tr><tr class="fragment" data-fragment-index="2"><td>3</td><td>0</td><td class="hl">Jon</td><td class="hl">9</td><td class="hl">10</td><td>30</td><td>.</td><td></td></tr><tr class="fragment" data-fragment-index="3"><td>3</td><td>0</td><td>Jon</td><td>9</td><td>10</td><td>30</td><td class="hl">19</td><td></td></tr><tr class="fragment" data-fragment-index="4"><td>3</td><td>0</td><td>Jon</td><td>9</td><td>10</td><td>30</td><td>19</td><td class="hl">Y</td></tr><tr class="fragment pdv-final" data-fragment-index="5"><td>3</td><td>0</td><td class="kc">Jon</td><td>9</td><td>10</td><td class="hl kc">49</td><td class="kc">19</td><td class="kc">Y</td></tr></table></div>
</div>
```

## Execution: row 4

Same steps, next row.

```{=html}
<div class="pdv-walk">
<div class="pdv-side"><pre class="pdv-code"><span class="pdv-line"><span class="kw">data</span> quiz_totals;<span class="fragment pdv-lit" data-fragment-index="1"></span></span><span class="pdv-line">    <span class="kw">set</span> scores;<span class="fragment pdv-lit" data-fragment-index="2"></span></span><span class="pdv-line">    <span class="kw">retain</span> Running 0;</span><span class="pdv-line">    Total = Quiz1 + Quiz2;<span class="fragment pdv-lit" data-fragment-index="3"></span></span><span class="pdv-line">    <span class="kw">if</span> Total &gt;= 15 <span class="kw">then</span> Pass = "Y";<span class="fragment pdv-lit" data-fragment-index="4"></span></span><span class="pdv-line">    <span class="kw">else</span> Pass = "N";<span class="fragment pdv-lit" data-fragment-index="5"></span></span><span class="pdv-line">    Running = <span class="kw">sum</span>(Running, Total);<span class="fragment pdv-lit" data-fragment-index="6"></span></span><span class="pdv-line">    <span class="kw">drop</span> Quiz1 Quiz2;</span><span class="pdv-line"><span class="kw">run</span>;<span class="fragment pdv-lit" data-fragment-index="7"></span></span></pre><div class="pdv-in"><div class="pdv-cap">Input: <code>work.scores</code></div><table class="mini pdv-intab"><tr><th>Student</th><th>Quiz1</th><th>Quiz2</th></tr><tr><td>Vince</td><td>8</td><td>9</td></tr><tr><td>Abi</td><td>6</td><td>7</td></tr><tr><td>Jon</td><td>9</td><td>10</td></tr><tr class="fragment pdv-inrow" data-fragment-index="2"><td>Maria</td><td>7</td><td>7</td></tr><tr><td>Kristian</td><td>10</td><td>.</td></tr></table></div><div class="out-table"><div class="pdv-cap">Output: <code>work.quiz_totals</code></div><table class="mini"><tr><th>Student</th><th>Running</th><th>Total</th><th>Pass</th></tr><tr><td>Vince</td><td>17</td><td>17</td><td>Y</td></tr><tr><td>Abi</td><td>30</td><td>13</td><td>N</td></tr><tr><td>Jon</td><td>49</td><td>19</td><td>Y</td></tr><tr class="fragment pdv-out-new" data-fragment-index="7"><td class="xfer">Maria</td><td class="xfer">63</td><td class="xfer">14</td><td class="xfer">N</td></tr></table></div></div>
<div class="pdv-wrap"><table class="pdv"><tr><th class="drop">_N_<span class="pdv-tag">D</span></th><th class="drop">_ERROR_<span class="pdv-tag">D</span></th><th class="keep">Student<span class="pdv-tag">K</span></th><th class="drop">Quiz1<span class="pdv-tag">D</span></th><th class="drop">Quiz2<span class="pdv-tag">D</span></th><th class="keep">Running<span class="pdv-tag">K</span></th><th class="keep">Total<span class="pdv-tag">K</span></th><th class="keep">Pass<span class="pdv-tag">K</span></th></tr><tr class="fragment" data-fragment-index="1"><td class="hl">4</td><td>0</td><td>Jon</td><td>9</td><td>10</td><td>49</td><td class="hl">.</td><td class="hl"></td></tr><tr class="fragment" data-fragment-index="2"><td>4</td><td>0</td><td class="hl">Maria</td><td class="hl">7</td><td class="hl">7</td><td>49</td><td>.</td><td></td></tr><tr class="fragment" data-fragment-index="3"><td>4</td><td>0</td><td>Maria</td><td>7</td><td>7</td><td>49</td><td class="hl">14</td><td></td></tr><tr class="fragment" data-fragment-index="4"><td>4</td><td>0</td><td>Maria</td><td>7</td><td>7</td><td>49</td><td class="hl">14</td><td></td></tr><tr class="fragment" data-fragment-index="5"><td>4</td><td>0</td><td>Maria</td><td>7</td><td>7</td><td>49</td><td>14</td><td class="hl">N</td></tr><tr class="fragment pdv-final" data-fragment-index="6"><td>4</td><td>0</td><td class="kc">Maria</td><td>7</td><td>7</td><td class="hl kc">63</td><td class="kc">14</td><td class="kc">N</td></tr></table></div>
</div>
```

## Execution: row 5

Kristian's `Quiz2` is missing. Watch what happens.

```{=html}
<div class="pdv-walk">
<div class="pdv-side"><pre class="pdv-code"><span class="pdv-line"><span class="kw">data</span> quiz_totals;<span class="fragment pdv-lit" data-fragment-index="1"></span></span><span class="pdv-line">    <span class="kw">set</span> scores;<span class="fragment pdv-lit" data-fragment-index="2"></span></span><span class="pdv-line">    <span class="kw">retain</span> Running 0;</span><span class="pdv-line">    Total = Quiz1 + Quiz2;<span class="fragment pdv-lit" data-fragment-index="3"></span></span><span class="pdv-line">    <span class="kw">if</span> Total &gt;= 15 <span class="kw">then</span> Pass = "Y";<span class="fragment pdv-lit" data-fragment-index="4"></span></span><span class="pdv-line">    <span class="kw">else</span> Pass = "N";<span class="fragment pdv-lit" data-fragment-index="5"></span></span><span class="pdv-line">    Running = <span class="kw">sum</span>(Running, Total);<span class="fragment pdv-lit" data-fragment-index="6"></span></span><span class="pdv-line">    <span class="kw">drop</span> Quiz1 Quiz2;</span><span class="pdv-line"><span class="kw">run</span>;<span class="fragment pdv-lit" data-fragment-index="7"></span></span></pre><div class="pdv-in"><div class="pdv-cap">Input: <code>work.scores</code></div><table class="mini pdv-intab"><tr><th>Student</th><th>Quiz1</th><th>Quiz2</th></tr><tr><td>Vince</td><td>8</td><td>9</td></tr><tr><td>Abi</td><td>6</td><td>7</td></tr><tr><td>Jon</td><td>9</td><td>10</td></tr><tr><td>Maria</td><td>7</td><td>7</td></tr><tr class="fragment pdv-inrow" data-fragment-index="2"><td>Kristian</td><td>10</td><td>.</td></tr></table></div><div class="out-table"><div class="pdv-cap">Output: <code>work.quiz_totals</code></div><table class="mini"><tr><th>Student</th><th>Running</th><th>Total</th><th>Pass</th></tr><tr><td>Vince</td><td>17</td><td>17</td><td>Y</td></tr><tr><td>Abi</td><td>30</td><td>13</td><td>N</td></tr><tr><td>Jon</td><td>49</td><td>19</td><td>Y</td></tr><tr><td>Maria</td><td>63</td><td>14</td><td>N</td></tr><tr class="fragment pdv-out-new" data-fragment-index="7"><td class="xfer">Kristian</td><td class="xfer">63</td><td class="xfer">.</td><td class="xfer">N</td></tr></table></div></div>
<div class="pdv-wrap"><table class="pdv"><tr><th class="drop">_N_<span class="pdv-tag">D</span></th><th class="drop">_ERROR_<span class="pdv-tag">D</span></th><th class="keep">Student<span class="pdv-tag">K</span></th><th class="drop">Quiz1<span class="pdv-tag">D</span></th><th class="drop">Quiz2<span class="pdv-tag">D</span></th><th class="keep">Running<span class="pdv-tag">K</span></th><th class="keep">Total<span class="pdv-tag">K</span></th><th class="keep">Pass<span class="pdv-tag">K</span></th></tr><tr class="fragment" data-fragment-index="1"><td class="hl">5</td><td>0</td><td>Maria</td><td>7</td><td>7</td><td>63</td><td class="hl">.</td><td class="hl"></td></tr><tr class="fragment" data-fragment-index="2"><td>5</td><td>0</td><td class="hl">Kristian</td><td class="hl">10</td><td class="hl">.</td><td>63</td><td>.</td><td></td></tr><tr class="fragment" data-fragment-index="3"><td>5</td><td>0</td><td>Kristian</td><td>10</td><td>.</td><td>63</td><td class="hl">.</td><td></td></tr><tr class="fragment" data-fragment-index="4"><td>5</td><td>0</td><td>Kristian</td><td>10</td><td>.</td><td>63</td><td class="hl">.</td><td></td></tr><tr class="fragment" data-fragment-index="5"><td>5</td><td>0</td><td>Kristian</td><td>10</td><td>.</td><td>63</td><td>.</td><td class="hl">N</td></tr><tr class="fragment pdv-final" data-fragment-index="6"><td>5</td><td>0</td><td class="kc">Kristian</td><td>10</td><td>.</td><td class="hl kc">63</td><td class="kc">.</td><td class="kc">N</td></tr></table></div>
</div>
```

::: {.fragment}
10 plus missing is missing, so `Total` is missing. A missing value is smaller than any number, so the `IF` is false and `Pass` is N. `sum()` ignores the missing value, so `Running` stays at 63.
:::

::: {.content-visible unless-format="revealjs"}
Kristian's `Total` is missing because we used a plus sign. If we had used `Total = sum(Quiz1, Quiz2);`, his total would be 10.

::: {.callout-tip collapse="true" title="Check yourself: what if Running used a plus sign too?"}
With `Running = Running + Total;`, Kristian's row would make `Running` missing (63 plus missing is missing). Because it's retained, it would stay missing for any rows after that.
:::
:::

::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
Is it fair to mark Kristian as failing? He got a 10 on the one quiz he took. SAS didn't make a decision about that. It just treated missing as less than 15. If we want missing values handled differently, we have to write that into our code, for example by checking `if Total = . then ...` first.
:::


## End of the step

SAS goes back to the top one more time, and `_N_` becomes 6. `SET` has no more rows to read, so the DATA step stops.

```{=html}
<div class="out-table"><div class="pdv-cap">Output: <code>work.quiz_totals</code></div><table class="mini"><tr><th>Student</th><th>Running</th><th>Total</th><th>Pass</th></tr><tr><td>Vince</td><td>17</td><td>17</td><td>Y</td></tr><tr><td>Abi</td><td>30</td><td>13</td><td>N</td></tr><tr><td>Jon</td><td>49</td><td>19</td><td>Y</td></tr><tr><td>Maria</td><td>63</td><td>14</td><td>N</td></tr><tr><td>Kristian</td><td>63</td><td>.</td><td>N</td></tr></table></div>
```

::: {.fragment}
5 rows in, 5 rows out. `Quiz1` and `Quiz2` were used in every row, but they aren't in the output.
:::


::: {.content-visible unless-format="revealjs"}
For the homework, use the same steps for each row:

1. At the top of the step, add 1 to `_N_`. Set new columns back to missing, unless they are retained.
2. `SET` copies the next row into the PDV.
3. Run each executable statement, top to bottom, and update the PDV.
4. At `run;`, write the row to the output table, leaving out anything marked `D`.
:::

## Important things to note

- Columns in a **`DROP`** statement are still in the PDV. We can use them for any calculation until the row is written to the output table.
- **`WHERE`** decides which rows go **into** the PDV.
- **`IF`** uses the values **in** the PDV right now.

::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
These three come up constantly, and mixing them up causes a lot of bugs. `DROP` controls what goes into the output table. `WHERE` controls what goes into the PDV. `IF` checks what is in the PDV at that moment.
:::


## `DROP` columns are still in the PDV

`Height` is dropped, but we can still use it to compute `BMI`.

In [ ]:
data drop_height;
    set sashelp.class;
    drop Height;    /* still enters the PDV, but not the output table */
    BMI = (Weight / Height**2) * 703;
run;

proc print data=drop_height (obs=5);
run;

Obs,Name,Sex,Age,Weight,BMI
1,Alfred,M,14,112.5,16.6115
2,Alice,F,13,84.0,18.4986
3,Barbara,F,13,98.0,16.1568
4,Carol,F,14,102.5,18.2709
5,Henry,M,14,102.5,17.8703


::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
`Height` is gone from the output, but `BMI` was computed correctly. Alfred's BMI is 16.6, using his height of 69 inches.

The `DROP` is even placed before the calculation. That doesn't matter, because `DROP` is declarative and takes effect during compilation. This is useful in practice. We can bring in helper columns, use them to compute what we need, and drop them so our output stays clean.
:::


## `WHERE` decides what enters the PDV

In [ ]:
data teenagers;
    set sashelp.class;
    where Age >= 14;    /* only these rows enter the PDV */
run;

proc print data=teenagers;
run;

Obs,Name,Sex,Age,Height,Weight
1,Alfred,M,14,69.0,112.5
2,Carol,F,14,62.8,102.5
3,Henry,M,14,63.5,102.5
4,Janet,F,15,62.5,112.5
5,Judy,F,14,64.3,90.0
6,Mary,F,15,66.5,112.0
7,Philip,M,16,72.0,150.0
8,Ronald,M,15,67.0,133.0
9,William,M,15,66.5,112.0


::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
Only 9 students are 14 or older, so only 9 rows ever enter the PDV. The younger students never get an iteration at all, so `_N_` only counts up to 9.

Because `WHERE` works before the row enters the PDV, it can only check columns that are already in the input table. We can't use `WHERE` on a column we create in the step, like `BMI`.
:::


## `IF` uses the PDV right now

The `IF` runs **before** we add 1 to `Age`. So it checks the original age.

In [ ]:
data example;
    set sashelp.class;
    if Age >= 14 then Teenager = 1;
    Age = Age + 1;    /* this change happens AFTER the IF */
run;

proc print data=example;
run;

Obs,Name,Sex,Age,Height,Weight,Teenager
1,Alfred,M,15,69.0,112.5,1
2,Alice,F,14,56.5,84.0,.
3,Barbara,F,14,65.3,98.0,.
4,Carol,F,15,62.8,102.5,1
5,Henry,M,15,63.5,102.5,1
6,James,M,13,57.3,83.0,.
7,Jane,F,13,59.8,84.5,.
8,Janet,F,16,62.5,112.5,1
9,Jeffrey,M,14,62.5,84.0,.
10,John,M,13,59.0,99.5,.


Look at Alice. Her `Age` now says 14, but `Teenager` is missing. She was 13 when the `IF` checked, and 1 was added after. Order matters!

::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
Barbara and Jeffrey also show 14 with a missing `Teenager`, for the same reason.

If we swapped the two lines, so `Age = Age + 1;` came first, the `IF` would see the new ages, and Alice, Barbara, and Jeffrey would get `Teenager = 1`. Which one is right depends on what we mean.

Also notice that `Teenager` is missing, not 0, for the younger students. We only set it when the `IF` is true, and there's no `ELSE`. To get 0, we would add `else Teenager = 0;`.
:::


## `_ERROR_`

In SAS, `_ERROR_` is set to 1 when certain data or processing problems happen in a DATA step. For example:

:::: {.columns}
::: {.column width="50%"}
- data conversion errors
- division by zero
- invalid math (like the log of a negative number)
- invalid array references
:::
::: {.column width="50%"}
- invalid date or time values
- errors in user-defined functions or formats
:::
::::

::: {.fragment}
**What happens when `_ERROR_` = 1?**

- a note or warning is written in the log
- the value is set to missing
- SAS keeps going with the next row
:::

::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
SAS won't stop our program for these problems. It quietly sets values to missing and keeps going. That's why we read the log, not just the output. If you see more missing values than you expected, check the log for notes about invalid data or division by zero.
:::


## `RETAIN` with BY groups

Recall: `RETAIN` leaves a value as it is in the PDV from one row to the next. Here is our food survey sorted by restaurant, with a running total from `retain Total 0;` and `Total = sum(Total, Score);`.

```{=html}
<table class="mini by-ctx"><tr><th>Student</th><th>Restaurant</th><th>Category</th><th>Score</th><th>Total</th><th class="fragment" data-fragment-index="1">What we want</th></tr><tr class="dots"><td colspan="6">&#8942;</td></tr><tr><td>Maria</td><td>Fuji</td><td>Service</td><td>7</td><td>63</td><td class="fragment" data-fragment-index="1">63</td></tr><tr><td>Kristian</td><td>Fuji</td><td>Food Quality</td><td>.</td><td>63</td><td class="fragment" data-fragment-index="1">63</td></tr><tr><td>Kristian</td><td>Fuji</td><td>Service</td><td>.</td><td>63</td><td class="fragment" data-fragment-index="1">63</td></tr><tr><td>Vince</td><td>Jamaican Grill</td><td>Food Quality</td><td>8</td><td class="hl">71</td><td class="fragment hl" data-fragment-index="1">8</td></tr><tr><td>Vince</td><td>Jamaican Grill</td><td>Service</td><td>6.5</td><td>77.5</td><td class="fragment" data-fragment-index="1">14.5</td></tr><tr class="dots"><td colspan="6">&#8942;</td></tr></table>
```

::: {.fragment fragment-index=2}
The total keeps going when Jamaican Grill starts. What if we want a total score **for each restaurant**?
:::


::: {.content-visible unless-format="revealjs"}
`RETAIN` alone gives one running total for the whole table. Jamaican Grill starts at 71 instead of 8, because Fuji's 63 points are carried along.
:::


::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
Why can't SAS just tell that a new restaurant started? The PDV only holds one row at a time. When SAS is working on Vince's Jamaican Grill row, it doesn't know the row before it was Fuji.
:::


## The `BY` statement

We need to reset the total when a new restaurant starts. A `BY` statement tells SAS how our rows are grouped, and creates two temporary columns in the PDV:

- `FIRST.Restaurant` is 1 on the **first** row of each restaurant, and 0 otherwise
- `LAST.Restaurant` is 1 on the **last** row of each restaurant, and 0 otherwise

```{=html}
<table class="mini by-ctx"><tr><th>Student</th><th>Restaurant</th><th>Category</th><th>Score</th><th>Total</th><th class="fragment" data-fragment-index="1">FIRST.Restaurant</th><th class="fragment" data-fragment-index="1">LAST.Restaurant</th></tr><tr class="dots"><td colspan="7">&#8942;</td></tr><tr><td>Maria</td><td>Fuji</td><td>Service</td><td>7</td><td>63</td><td class="fragment" data-fragment-index="1">0</td><td class="fragment" data-fragment-index="1">0</td></tr><tr><td>Kristian</td><td>Fuji</td><td>Food Quality</td><td>.</td><td>63</td><td class="fragment" data-fragment-index="1">0</td><td class="fragment" data-fragment-index="1">0</td></tr><tr><td>Kristian</td><td>Fuji</td><td>Service</td><td>.</td><td>63</td><td class="fragment" data-fragment-index="1">0</td><td class="fragment hl" data-fragment-index="1">1</td></tr><tr><td>Vince</td><td>Jamaican Grill</td><td>Food Quality</td><td>8</td><td>71</td><td class="fragment hl" data-fragment-index="1">1</td><td class="fragment" data-fragment-index="1">0</td></tr><tr><td>Vince</td><td>Jamaican Grill</td><td>Service</td><td>6.5</td><td>77.5</td><td class="fragment" data-fragment-index="1">0</td><td class="fragment" data-fragment-index="1">0</td></tr><tr class="dots"><td colspan="7">&#8942;</td></tr></table>
```

::: {.fragment fragment-index=2}
When `FIRST.Restaurant` is 1, we set the total back to 0.
:::


::: {.content-visible unless-format="revealjs"}
Like `_N_` and `_ERROR_`, `FIRST.` and `LAST.` columns are only in the PDV. They are never written to the output table.
:::


::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
The period between `FIRST` and the column name is part of the name. We get one `FIRST.` and one `LAST.` column for each column in the `BY` statement.
:::


## Step 1: sort by the group

`BY` only works if the data is sorted by that column.

In [ ]:
libname ma505 "~";    /* skip if your autoexec does this */

/* first step is to sort the data by the group */
proc sort data=ma505.food_surv out=food_surv_rest_sort;
    by Restaurant;
run;

::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
SAS finds where groups start and end by comparing each row to the next one. If the Fuji rows aren't next to each other, that doesn't work, and SAS stops with an error saying the data is not sorted.

After sorting, all 10 Fuji rows come first, then 10 Jamaican Grill rows, then 10 Kings rows. If you get an error that `ma505.food_surv` doesn't exist, rerun the import of `food_surv.csv` (with `guessingrows=max;`) first.
:::


## Step 2: total by restaurant

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="kw">data</span> sum_by_group;
    <span class="kw">set</span> food_surv_rest_sort;
    <span class="libref-circle"><span class="kw">by</span> Restaurant;</span><span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">creates <span class="mono">First.Restaurant</span> and <span class="mono">Last.Restaurant</span> in the PDV</span></span>
    <span class="kw">retain</span> RestaurantSum 0;
    <span class="libref-circle"><span class="kw">if</span> First.Restaurant = 1 <span class="kw">then</span> RestaurantSum = 0;</span><span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">start over at 0 for each new restaurant</span></span>
    RestaurantSum = <span class="kw">sum</span>(RestaurantSum, Score);
    <span class="libref-circle"><span class="kw">if</span> Last.Restaurant = 1;</span><span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">only writes the last row of each restaurant</span></span>
<span class="kw">run</span>;</pre></div></div>
```

::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
`if Last.Restaurant = 1;` has no `THEN`. This is called a **subsetting IF**. If the condition is true, SAS keeps going with this row. If it's false, SAS stops right there, goes back to the top, and doesn't write the row out. So only the last row of each restaurant gets written, and on that row `RestaurantSum` holds the full total.

Why not `WHERE`? `LAST.Restaurant` only exists in the PDV, so `WHERE` can't see it. `WHERE` filters rows before they enter the PDV. A subsetting `IF` filters rows after they're in the PDV, so it can use anything we've computed.
:::


## In the PDV

```{=html}
<div class="pdv-walk wide">
<pre class="pdv-code"><span class="pdv-line"><span class="kw">data</span> sum_by_group;</span><span class="pdv-line">    <span class="kw">set</span> food_surv_rest_sort;<span class="fragment pdv-lit" data-fragment-index="1"></span><span class="fragment pdv-lit" data-fragment-index="4"></span></span><span class="pdv-line">    <span class="kw">by</span> Restaurant;<span class="fragment pdv-lit" data-fragment-index="1"></span></span><span class="pdv-line">    <span class="kw">retain</span> RestaurantSum 0;</span><span class="pdv-line">    <span class="kw">if</span> First.Restaurant = 1 <span class="kw">then</span> RestaurantSum = 0;<span class="fragment pdv-lit" data-fragment-index="4"></span></span><span class="pdv-line">    RestaurantSum = <span class="kw">sum</span>(RestaurantSum, Score);<span class="fragment pdv-lit" data-fragment-index="2"></span><span class="fragment pdv-lit" data-fragment-index="5"></span></span><span class="pdv-line">    <span class="kw">if</span> Last.Restaurant = 1;<span class="fragment pdv-lit" data-fragment-index="3"></span></span><span class="pdv-line"><span class="kw">run</span>;</span></pre>
<div class="pdv-in"><div class="pdv-cap">Input: <code>work.food_surv_rest_sort</code></div><table class="mini pdv-intab"><tr><th>Row</th><th>Student</th><th>Restaurant</th><th>Category</th><th>Score</th></tr><tr class="fragment pdv-inrow" data-fragment-index="1"><td>1</td><td>Vince</td><td>Fuji</td><td>Food Quality</td><td>9</td></tr><tr class="dots"><td colspan="5">&#8942;</td></tr><tr class="fragment pdv-inrow" data-fragment-index="3"><td>10</td><td>Kristian</td><td>Fuji</td><td>Service</td><td>.</td></tr><tr class="fragment pdv-inrow" data-fragment-index="4"><td>11</td><td>Vince</td><td>Jamaican Grill</td><td>Food Quality</td><td>8</td></tr><tr class="dots"><td colspan="5">&#8942;</td></tr></table></div>
<div class="pdv-wrap">
<table class="pdv"><tr><th class="drop">_N_<span class="pdv-tag">D</span></th><th class="drop">_ERROR_<span class="pdv-tag">D</span></th><th class="keep">Student<span class="pdv-tag">K</span></th><th class="keep">Restaurant<span class="pdv-tag">K</span></th><th class="keep">Category<span class="pdv-tag">K</span></th><th class="keep">Score<span class="pdv-tag">K</span></th><th class="keep">RestaurantSum<span class="pdv-tag">K</span></th><th class="drop">First.Restaurant<span class="pdv-tag">D</span></th><th class="drop">Last.Restaurant<span class="pdv-tag">D</span></th></tr>
<tr class="fragment" data-fragment-index="1"><td>1</td><td>0</td><td class="hl">Vince</td><td class="hl">Fuji</td><td class="hl">Food Quality</td><td class="hl">9</td><td>0</td><td class="hl">1</td><td class="hl">0</td></tr>
<tr class="fragment" data-fragment-index="2"><td>1</td><td>0</td><td>Vince</td><td>Fuji</td><td>Food Quality</td><td>9</td><td class="hl">9</td><td>1</td><td>0</td></tr>
<tr class="fragment dots" data-fragment-index="3"><td colspan="9">&#8942;</td></tr><tr class="fragment" data-fragment-index="3"><td>10</td><td>0</td><td>Kristian</td><td>Fuji</td><td>Service</td><td>.</td><td class="hl">63</td><td>0</td><td class="hl">1</td></tr>
<tr class="fragment" data-fragment-index="4"><td>11</td><td>0</td><td class="hl">Vince</td><td class="hl">Jamaican Grill</td><td class="hl">Food Quality</td><td class="hl">8</td><td class="hl">0</td><td class="hl">1</td><td>0</td></tr>
<tr class="fragment" data-fragment-index="5"><td>11</td><td>0</td><td>Vince</td><td>Jamaican Grill</td><td>Food Quality</td><td>8</td><td class="hl">8</td><td>1</td><td>0</td></tr>
</table></div>
</div>
```

::: {.fragment}
Row 10 is the last Fuji row (`Last.Restaurant` = 1), so it is written with a total of 63. On row 11, `First.Restaurant` = 1, so the total starts over.
:::

In [ ]:
data sum_by_group;
    set food_surv_rest_sort;
    by Restaurant;
    retain RestaurantSum 0;
    if First.Restaurant = 1 then RestaurantSum = 0;
    RestaurantSum = sum(RestaurantSum, Score);
    if Last.Restaurant = 1;    /* only outputs the last row of each group */
run;

proc print data=sum_by_group;
run;

Obs,Student,Restaurant,Category,Score,RestaurantSum
1,Kristian,Fuji,Service,.,63.0
2,Kristian,Jamaican Grill,Service,.,60.5
3,Kristian,Kings,Service,.,60.0


`Student`, `Category`, and `Score` are just the values from the last row of each restaurant, so we would usually drop them.

::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
Rows 2 through 9 are similar. `FIRST.Restaurant` and `LAST.Restaurant` are both 0, the total keeps building, and no rows are written.

A nice check: 63 + 60.5 + 60 = 183.5, the same grand total as the running total over the whole table.

To keep just the totals, we would add `drop Student Category Score;`. This pattern comes up again and again: sort, then `BY`, `RETAIN`, reset on `FIRST.`, and output on `LAST.`. Any time we need a total, a count, or a summary for each group in a DATA step, this is the recipe.
:::


## Merging tables

```{=html}
<div class="syntax-box"><b>DATA</b> <i>output-table</i>;<br>&nbsp;&nbsp;&nbsp;&nbsp;<b>MERGE</b> <i>input-table1 input-table2</i> ...;<br>&nbsp;&nbsp;&nbsp;&nbsp;<b>BY</b> <i>BY-column(s)</i>;<br><b>RUN;</b></div>
```

`MERGE` puts tables together **side by side**, matching rows on the `BY` columns.

Import `class_all.csv` and `class_teachers.csv` into your `work` library first.

In [ ]:
proc import datafile="~/class_all.csv" out=class_all dbms=csv replace;
run;

proc import datafile="~/class_teachers.csv" out=teacher dbms=csv replace;
run;

proc print data=teacher (obs=5);
run;

proc print data=class_all (obs=5);
run;

The Print Procedure 
 
 
 
 
 Obs 
 Name 
 Grade 
 Teacher 
 
 
 
 
 1 
 Sam Patel 
 Junior 
 Mr. Ortiz 
 
 
 2 
 Alex Smith 
 Junior 
 Ms. Foster 
 
 
 3 
 Taylor Lee 
 Senior 
 Dr. Kim 
 
 
 4 
 Jordan Brown 
 Freshman 
 Ms. Foster 
 
 
 5 
 Casey Nguyen 
 Senior 
 Dr. Kim 
 
 
 
 The Print Procedure 
 
 
 
 
 Obs 
 Name 
 Age 
 Weight 
 Height 
 Grade 
 GPA 
 
 
 
 
 1 
 Taylor Lee 
 21 
 132.4 
 5.4 
 Senior 
 3.65 
 
 
 2 
 Jordan Brown 
 19 
 170.3 
 6 
 Freshman 
 2.89 
 
 
 3 
 Morgan Davis 
 20 
 155 
 5.7 
 Sophomore 
 3.42 
 
 
 4 
 Casey Nguyen 
 22 
 140.8 
 5.5 
 Senior 
 3.91 
 
 
 5 
 Riley Garcia 
 19 
 182.1 
 6.1 
 Freshman 
 3.05

::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
Using `SET` with two tables stacks them, adding more **rows**. `MERGE` puts them side by side, adding more **columns**.

The two tables have `Name` and `Grade` in common, so that's how we match them. The rows are in a different order in each table (Sam is first in one, Taylor in the other), so we can't just glue them side by side.
:::


## Step 1: sort both tables

Just like `BY` groups, both tables need to be sorted by the `BY` columns.

In [ ]:
proc sort data=class_all;
    by Name Grade;
run;

proc sort data=teacher;
    by Name Grade;
run;

::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
There's no `OUT=` here, so each table is sorted in place. That's fine, because these are our working copies in `work`.

Why sort and merge by both `Name` and `Grade`? Two students could have the same name. Matching on both makes it much less likely we match the wrong two people.
:::


## Step 2: merge

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="cmt">/* merge together */</span>
<span class="kw">data</span> class_merge;
    <span class="libref-circle"><span class="kw">merge</span> class_all teacher;</span><span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">merging the <span class="mono">class_all</span> table and the <span class="mono">teacher</span> table</span></span>
    <span class="kw">by</span> Name Grade;
<span class="kw">run</span>;</pre></div></div>
```

In [ ]:
data class_merge;
    merge class_all teacher;
    by Name Grade;
run;

proc print data=class_merge;
run;

Obs,Name,Age,Weight,Height,Grade,GPA,Teacher
1,Alex Smith,20,145.6,5.8,Junior,3.27,Ms. Foster
2,Casey Nguyen,22,140.8,5.5,Senior,3.91,Dr. Kim
3,Jamie Wilson,20,165.7,5.9,Sophomore,2.74,Mr. Ortiz
4,Jordan Brown,19,170.3,6,Freshman,2.89,Ms. Foster
5,Morgan Davis,20,155,5.7,Sophomore,3.42,Mr. Ortiz
6,Riley Garcia,19,182.1,6.1,Freshman,3.05,Ms. Foster
7,Sam Patel,21,128.9,5.3,Junior,3.58,Mr. Ortiz
8,Taylor Lee,21,132.4,5.4,Senior,3.65,Dr. Kim


::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
Two things to know for real data:

- If a student is in one table but not the other, SAS still includes them, with missing values for the columns from the table they're missing from. After a merge, check for unexpected missing values.
- If both tables have a column with the same name that isn't in the `BY` statement, the value from the second table overwrites the first. It's worth checking both tables with `PROC CONTENTS` before you merge.
:::


## How merge uses the PDV

**Compilation:** the columns from the **first** table go into the PDV, then any new columns from the second table.

```{=html}
<div class="pdv-walk wide">
<pre class="pdv-code"><span class="pdv-line"><span class="kw">data</span> class_merge;</span><span class="pdv-line">    <span class="kw">merge</span> class_all teacher;<span class="fragment pdv-lit" data-fragment-index="1"></span><span class="fragment pdv-lit" data-fragment-index="2"></span><span class="fragment pdv-lit" data-fragment-index="3"></span><span class="fragment pdv-lit" data-fragment-index="4"></span></span><span class="pdv-line">    <span class="kw">by</span> Name Grade;<span class="fragment pdv-lit" data-fragment-index="3"></span><span class="fragment pdv-lit" data-fragment-index="4"></span></span><span class="pdv-line"><span class="kw">run</span>;</span></pre>
<div class="pdv-ins"><div class="pdv-in"><div class="pdv-cap">Input: <code>class_all</code></div><table class="mini pdv-intab"><tr><th>Name</th><th>Age</th><th>Weight</th><th>Height</th><th>Grade</th><th>GPA</th></tr><tr class="fragment pdv-inrow" data-fragment-index="3"><td>Alex Smith</td><td>20</td><td>145.6</td><td>5.8</td><td>Junior</td><td>3.27</td></tr><tr><td>Casey Nguyen</td><td>22</td><td>140.8</td><td>5.5</td><td>Senior</td><td>3.91</td></tr><tr class="dots"><td colspan="6">&#8942;</td></tr></table></div><div class="pdv-in"><div class="pdv-cap">Input: <code>teacher</code></div><table class="mini pdv-intab"><tr><th>Name</th><th>Grade</th><th>Teacher</th></tr><tr class="fragment pdv-inrow" data-fragment-index="4"><td>Alex Smith</td><td>Junior</td><td>Ms. Foster</td></tr><tr><td>Casey Nguyen</td><td>Senior</td><td>Dr. Kim</td></tr><tr class="dots"><td colspan="3">&#8942;</td></tr></table></div></div>
<div class="pdv-wrap"><table class="pdv"><thead><tr><th class="drop fragment" data-fragment-index="1">_N_<span class="pdv-tag">D</span></th><th class="drop fragment" data-fragment-index="1">_ERROR_<span class="pdv-tag">D</span></th><th class="keep fragment" data-fragment-index="1">Name<span class="pdv-tag">K</span></th><th class="keep fragment" data-fragment-index="1">Age<span class="pdv-tag">K</span></th><th class="keep fragment" data-fragment-index="1">Weight<span class="pdv-tag">K</span></th><th class="keep fragment" data-fragment-index="1">Height<span class="pdv-tag">K</span></th><th class="keep fragment" data-fragment-index="1">Grade<span class="pdv-tag">K</span></th><th class="keep fragment" data-fragment-index="1">GPA<span class="pdv-tag">K</span></th><th class="keep fragment" data-fragment-index="2">Teacher<span class="pdv-tag">K</span></th></tr></thead><tr class="fragment" data-fragment-index="3"><td>1</td><td>0</td><td class="hl">Alex Smith</td><td class="hl">20</td><td class="hl">145.6</td><td class="hl">5.8</td><td class="hl">Junior</td><td class="hl">3.27</td><td></td></tr><tr class="fragment" data-fragment-index="4"><td>1</td><td>0</td><td>Alex Smith</td><td>20</td><td>145.6</td><td>5.8</td><td>Junior</td><td>3.27</td><td class="hl">Ms. Foster</td></tr></table></div>
</div>
```

::: {.fragment fragment-index=3}
**Execution:** SAS reads the rows from both tables in order. When the `BY` values match, the values from both go into the same row of the PDV.
:::


::: {.content-visible unless-format="revealjs"}
`Name` and `Grade` are in both tables, so they only go into the PDV once. `Teacher` is the only new column from `teacher`, so it's added last.
:::


## One-to-many

::::: {.fragment fragment-index=1}
What happens when one student has **many** rows in the other table?

:::: {.columns}
::: {.column width="48%"}
```{=html}
<table class="mini"><tr><th>Name</th><th>Grade</th><th>Age</th><th>GPA</th></tr>
<tr><td>Alex Smith</td><td>Junior</td><td>20</td><td>3.27</td></tr>
<tr><td>Jordan Brown</td><td>Freshman</td><td>19</td><td>2.89</td></tr></table>
```
:::
::: {.column width="4%"}
:::
::: {.column width="48%"}
```{=html}
<table class="mini"><tr><th>Name</th><th>Grade</th><th>Subject</th><th>Test Score</th></tr>
<tr><td>Alex Smith</td><td>Junior</td><td>Math</td><td>98</td></tr>
<tr><td>Alex Smith</td><td>Junior</td><td>English</td><td>94</td></tr>
<tr><td>Jordan Brown</td><td>Freshman</td><td>Math</td><td>89</td></tr></table>
```
:::
::::
:::::

::: {.fragment fragment-index=2}
We merge `BY Name Grade`, and we still need to sort both tables first.

The result has **one row per test**. Alex's age and GPA are repeated on both of his rows, because SAS keeps his values in the PDV while it reads his next test.
:::

::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
When each student has exactly one row in each table, it's called a **one-to-one** merge. Here, `class_all` has one row per student, and `tests` has one row per test.
:::


## One-to-many in the PDV

Let's follow Alex. He has 1 row in `class_all` and 2 rows in `tests`.

```{=html}
<div class="pdv-walk wide">
<pre class="pdv-code"><span class="pdv-line"><span class="kw">data</span> class_tests;</span><span class="pdv-line">    <span class="kw">merge</span> class_all tests;<span class="fragment pdv-lit" data-fragment-index="1"></span><span class="fragment pdv-lit" data-fragment-index="3"></span></span><span class="pdv-line">    <span class="kw">by</span> Name Grade;</span><span class="pdv-line"><span class="kw">run</span>;<span class="fragment pdv-lit" data-fragment-index="2"></span><span class="fragment pdv-lit" data-fragment-index="4"></span></span></pre>
<div class="pdv-ins"><div class="pdv-in"><div class="pdv-cap">Input: <code>class_all</code></div><table class="mini pdv-intab"><tr><th>Name</th><th>Age</th><th>Weight</th><th>Height</th><th>Grade</th><th>GPA</th></tr><tr class="fragment pdv-inrow" data-fragment-index="1"><td>Alex Smith</td><td>20</td><td>145.6</td><td>5.8</td><td>Junior</td><td>3.27</td></tr><tr><td>Casey Nguyen</td><td>22</td><td>140.8</td><td>5.5</td><td>Senior</td><td>3.91</td></tr><tr class="dots"><td colspan="6">&#8942;</td></tr></table></div><div class="pdv-in"><div class="pdv-cap">Input: <code>tests</code></div><table class="mini pdv-intab"><tr><th>Name</th><th>Grade</th><th>Subject</th><th>TestScore</th></tr><tr class="fragment pdv-inrow" data-fragment-index="1"><td>Alex Smith</td><td>Junior</td><td>Math</td><td>98</td></tr><tr class="fragment pdv-inrow" data-fragment-index="3"><td>Alex Smith</td><td>Junior</td><td>English</td><td>94</td></tr><tr><td>Casey Nguyen</td><td>Senior</td><td>English</td><td>88</td></tr><tr class="dots"><td colspan="4">&#8942;</td></tr></table></div></div>
<div class="pdv-wrap"><table class="pdv"><tr><th class="drop">_N_<span class="pdv-tag">D</span></th><th class="drop">_ERROR_<span class="pdv-tag">D</span></th><th class="keep">Name<span class="pdv-tag">K</span></th><th class="keep">Age<span class="pdv-tag">K</span></th><th class="keep">Weight<span class="pdv-tag">K</span></th><th class="keep">Height<span class="pdv-tag">K</span></th><th class="keep">Grade<span class="pdv-tag">K</span></th><th class="keep">GPA<span class="pdv-tag">K</span></th><th class="keep">Subject<span class="pdv-tag">K</span></th><th class="keep">TestScore<span class="pdv-tag">K</span></th></tr><tr class="fragment" data-fragment-index="1"><td>1</td><td>0</td><td class="hl">Alex Smith</td><td class="hl">20</td><td class="hl">145.6</td><td class="hl">5.8</td><td class="hl">Junior</td><td class="hl">3.27</td><td class="hl">Math</td><td class="hl">98</td></tr><tr class="fragment" data-fragment-index="3"><td class="hl">2</td><td>0</td><td>Alex Smith</td><td>20</td><td>145.6</td><td>5.8</td><td>Junior</td><td>3.27</td><td class="hl">English</td><td class="hl">94</td></tr></table></div>
<div class="out-table"><div class="pdv-cap">Output: <code>class_tests</code></div><table class="mini"><tr><th>Name</th><th>Age</th><th>Weight</th><th>Height</th><th>Grade</th><th>GPA</th><th>Subject</th><th>TestScore</th></tr><tr class="fragment" data-fragment-index="2"><td class="hl">Alex Smith</td><td class="hl">20</td><td class="hl">145.6</td><td class="hl">5.8</td><td class="hl">Junior</td><td class="hl">3.27</td><td class="hl">Math</td><td class="hl">98</td></tr><tr class="fragment" data-fragment-index="4"><td class="hl">Alex Smith</td><td class="hl">20</td><td class="hl">145.6</td><td class="hl">5.8</td><td class="hl">Junior</td><td class="hl">3.27</td><td class="hl">English</td><td class="hl">94</td></tr></table></div>
</div>
```

::: {.fragment fragment-index=5}
Alex has no more rows in `class_all`, so SAS keeps his values in the PDV. Only `Subject` and `TestScore` change.
:::


::: {.content-visible unless-format="revealjs"}
This is different from a regular DATA step, where values are set back to missing at the top of each iteration. In a merge, values from a table stay in the PDV until SAS reads a new row from that same table. When Casey's rows come up, SAS reads new rows from both tables, and the PDV fills with her values.
:::


## One-to-many in SAS

Import `class_tests.csv`, sort it, and merge it with `class_all`.


In [ ]:
proc import datafile="~/class_tests.csv" out=tests dbms=csv replace;
run;

proc sort data=tests;
    by Name Grade;
run;

data class_tests;
    merge class_all tests;
    by Name Grade;
run;

proc print data=class_tests;
run;

Obs,Name,Age,Weight,Height,Grade,GPA,Subject,TestScore
1,Alex Smith,20,145.6,5.8,Junior,3.27,Math,98
2,Alex Smith,20,145.6,5.8,Junior,3.27,English,94
3,Casey Nguyen,22,140.8,5.5,Senior,3.91,English,88
4,Casey Nguyen,22,140.8,5.5,Senior,3.91,Math,95
5,Jamie Wilson,20,165.7,5.9,Sophomore,2.74,English,81
6,Jordan Brown,19,170.3,6,Freshman,2.89,Math,89
7,Jordan Brown,19,170.3,6,Freshman,2.89,English,76
8,Morgan Davis,20,155,5.7,Sophomore,3.42,English,85
9,Riley Garcia,19,182.1,6.1,Freshman,3.05,Math,72
10,Sam Patel,21,128.9,5.3,Junior,3.58,Math,90


::: {.content-visible unless-format="revealjs"}
<!-- notes8 -->
The output has 11 rows. Alex, Casey, and Jordan each have two tests, so they each get two rows, with their information repeated. Jamie, Morgan, Riley, Sam, and Taylor have one test each.

This pattern is really common: customers and their orders, patients and their visits, players and their games. One table describes the people, the other describes the events, and merging puts each event next to the person's information.

::: {.callout-tip collapse="true" title="Check yourself: why is Kristian's Pass N in the quiz example?"}
His `Total` was missing, and SAS treats missing as smaller than any number, so `Total >= 15` was false.
:::

::: {.callout-tip collapse="true" title="Check yourself: what is the difference between WHERE and a subsetting IF?"}
`WHERE` filters before rows enter the PDV, so it only sees columns from the input table. A subsetting `IF` filters inside the PDV, so it can use new and temporary columns like `LAST.Restaurant`.
:::

::: {.callout-tip collapse="true" title="Check yourself: what must we do before using BY?"}
Sort the data by the `BY` columns, in a DATA step with `BY` and before a merge.
:::

::: {.callout-tip collapse="true" title="Check yourself: what is the difference between SET with two tables and MERGE?"}
`SET` stacks rows on top of each other. `MERGE` puts columns side by side, matched by the `BY` columns.
:::
:::
